# backwardCGM-PD — Simulation B, p=8 trên Kaggle

Job độc lập cho Scenario B, p=8, gồm 20 replicate và cả hai phương pháp. Hãy **Add Input** dataset `backwardCGM-PD`.

In [ ]:
import importlib.util, subprocess, sys
required = {"rdata": "rdata>=0.11", "networkx": "networkx>=3.0", "joblib": "joblib>=1.3"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import pandas as pd
from IPython.display import Image, display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/backwardCGM-PD")
RESULTS = Path("/kaggle/working/simulation-B-p8-results")
RESULTS.mkdir(parents=True, exist_ok=True)

# Khôi phục checkpoint từ output của một Kaggle Version trước nếu đã Add Input.
checkpoint_archives = list(INPUT_ROOT.rglob("simulation-B-p8-results.zip"))
checkpoint_files = list(INPUT_ROOT.rglob("simulation-B-p8.json"))
if checkpoint_archives:
    with zipfile.ZipFile(checkpoint_archives[0]) as archive:
        archive.extractall(RESULTS)
    print("Restored checkpoint:", checkpoint_archives[0])
elif checkpoint_files:
    shutil.copytree(checkpoint_files[0].parent, RESULTS, dirs_exist_ok=True)
    print("Restored checkpoint:", checkpoint_files[0])

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/simulation.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")

PORT_ROOT = WORK_ROOT / "python-port"
data_files = [
    WORK_ROOT / f"simulation/simulated-data/simdf_{s}_{p}.RData"
    for s in ("11", "22") for p in (8, 12, 16, 20)
]
missing_files = [str(p) for p in data_files if not p.exists()]
if not (PORT_ROOT / "experiments/simulation.py").exists() or missing_files:
    raise FileNotFoundError("Dataset Simulation không đầy đủ: " + str(missing_files))
print("Dataset: OK\nOutput:", RESULTS)

## Cấu hình cố định

Job này chạy Scenario B, p=8, 20 replicate và cả `tau` lẫn `submodel`. Mỗi replicate hoàn tất sẽ được checkpoint ngay.

In [ ]:
SCENARIO = "B"
P = "8"
REPLICATES = 20
METHOD = "both"

print({"scenario": SCENARIO, "p": P, "replicates": REPLICATES, "method": METHOD})

In [ ]:
output = RESULTS / "simulation-B-p8.json"
command = [
    sys.executable, "-u", str(PORT_ROOT / "experiments/simulation.py"),
    "--scenario", SCENARIO, "--p", P,
    "--replicates", str(REPLICATES), "--method", METHOD,
    "--source", "saved", "--alpha", "0.05", "--itmax", "500",
    "--rcon-backend", "grc_ipms", "--parallel", "3",
    "--output", str(output), "--resume",
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=PORT_ROOT, check=True)

In [ ]:
summary = output.with_name(f"{output.stem}-summary.csv")
if summary.exists():
    display(pd.read_csv(summary))
for scenario in ("A", "B"):
    figure = output.with_name(f"{output.stem}-scenario-{scenario}.png")
    if figure.exists():
        display(Image(filename=str(figure)))
archive = shutil.make_archive("/kaggle/working/simulation-B-p8-results", "zip", root_dir=RESULTS)
print("Download:", archive)

**Backend:** cả `tau` và `submodel` fit candidate models bằng Python port của `gRc::rcox(method="ipms")`, với 3 workers như R gốc.

**Checkpoint:** JSON được ghi atomically sau mỗi replicate. Checkpoint version 3 không resume kết quả MLE version 2; nếu chạy session mới, hãy **Save Version** rồi **Add Input** output IPMS trước đó.

**Blocker còn lại:** repo thiếu source `backwardCGMpd1()`, nên bookkeeping `#models` của method `tau` không thể khôi phục tuyệt đối.